In [3]:
# ============================================================
# BUILD DIM_CONSTRUCTORS — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# p_batch_id = "20250101_090000"

def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/04_gold_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [4]:
# -----------------------------------------
# p_batch_id
# -----------------------------------------
if "p_batch_id" in globals() and p_batch_id not in [None, ""]:
    pass
elif "dbutils" in globals():
    p_batch_id = dbutils.widgets.get("p_batch_id")
else:
    raise Exception("❌ p_batch_id not provided")

print("Using p_batch_id:", p_batch_id)

Exception: ❌ p_batch_id not provided

In [ ]:
# -----------------------------------------
# Paths
# -----------------------------------------
silver_constructors = f"{SILVER_PATH}/constructors/data.parquet"
gold_ref = f"{GOLD_PATH}/ref_nationality_region/data.parquet"

gold_path = f"{GOLD_PATH}/dim_constructors"
os.makedirs(gold_path, exist_ok=True)

In [ ]:
# -----------------------------------------
# Read Silver + Gold reference
# -----------------------------------------
constructors_df = spark.read.parquet(silver_constructors).filter(F.col("batch_id") == p_batch_id)
ref_df = spark.read.parquet(gold_ref)

if constructors_df.count() == 0:
    raise Exception("❌ Silver constructors has 0 rows for this batch_id")

In [ ]:
# -----------------------------------------
# Join
# -----------------------------------------
dim_constructors_df = (
    constructors_df.alias("c")
        .join(
            ref_df.alias("r"),
            F.col("c.nationality") == F.col("r.nationality"),
            "left"
        )
        .select(
            F.col("c.constructor_id"),
            F.col("c.constructor_name"),
            F.col("c.nationality"),
            F.col("r.region").alias("nationality_region")
        )
)

In [ ]:
# -----------------------------------------
# Write
# -----------------------------------------
write_to_gold(
    input_df=dim_constructors_df,
    target_path=f"{gold_path}/data.parquet",
    merge_keys=["constructor_id"]
)

print("✔ dim_constructors written")

spark.read.parquet(f"{gold_path}/data.parquet").show(5, truncate=False)

✔ dim_constructors written to: /Users/manoharazalki/F1-ANALYTICS/data/gold/dim_constructors/dim_constructors.parquet
+--------------+----------------+-------------+------------------+
|constructor_id|constructor_name|nationality  |nationality_region|
+--------------+----------------+-------------+------------------+
|adams         |Adams           |American     |North America     |
|afm           |AFM             |German       |Europe            |
|ags           |AGS             |French       |Europe            |
|alfa          |Alfa Romeo      |Swiss        |Europe            |
|alphatauri    |AlphaTauri      |Italian      |Europe            |
|alpine        |Alpine F1 Team  |French       |Europe            |
|alta          |Alta            |British      |Europe            |
|amon          |Amon            |New Zealander|Oceania           |
|apollon       |Apollon         |Swiss        |Europe            |
|arrows        |Arrows          |British      |Europe            |
+-----------